# Enterprise Semantic Search & Vector Retrieval with Pinecone

This project implements an end-to-end vector search engine using **Pinecone Serverless Vector Database** and **Sentence Transformers** (`all-MiniLM-L6-v2`). It indexes curriculum and technical course modules with weighted multi-field embeddings, rich metadata payloads, and hierarchical search capabilities.

### Architecture Overview
1. **Vector Database:** Pinecone Serverless (AWS `us-east-1`, Cosine Similarity Metric, 384 Dimensions)
2. **Embedding Model:** SentenceTransformers (`all-MiniLM-L6-v2`)
3. **Multi-Field Feature Weighting:** Differential weighting for Course Title (5x), Section Name (3x), Description (2x), and Technology (1x)
4. **Metadata Indexing:** Payload storage for dynamic filtering and instant result reconstruction without secondary database lookups
5. **Hierarchical Search:** Fine-grained section search in default namespace and aggregated course-level search in dedicated namespace

## 1. Environment Setup & Library Imports

In [189]:
import os
import pprint
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

# Load environment variables from .env file
load_dotenv(find_dotenv(), override=True)
print("Environment initialized successfully.")

Environment initialized successfully.


## 2. Initialize Model & Pinecone Vector Database

In [190]:
# Initialize 384-dimensional dense embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dimension = model.get_sentence_embedding_dimension()

# Initialize Pinecone Client
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

index_name = "my-index"
existing_indexes = [idx.name for idx in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=embedding_dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{index_name}' created successfully.")
else:
    print(f"Index '{index_name}' already exists. Connecting to existing index.")

index = pc.Index(index_name)
print("Pinecone connection verified:", index.describe_index_stats())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1677.76it/s]
C:\Users\Hp\AppData\Local\Temp\ipykernel_256\807069347.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


Index 'my-index' already exists. Connecting to existing index.
Pinecone connection verified: DescribeIndexStatsResponse(dimension=384, total_vector_count=680, metric='cosine', namespaces=1)


## 3. Data Ingestion & Metadata Construction

Load curriculum descriptions, assign unique composite identifiers (`course_id-section_id`), and prepare structured metadata dictionaries for Pinecone payload storage.

In [191]:
files = pd.read_csv("course_section_descriptions.csv", encoding="ANSI")

# Construct unique record IDs
files["unique_id"] = files["course_id"].astype(str) + "-" + files["section_id"].astype(str)

# Construct metadata payloads
files["metadata"] = files.apply(lambda row: {
    "course_id": int(row["course_id"]),
    "course_name": str(row["course_name"]),
    "section_name": str(row["section_name"]),
    "section_description": str(row["section_description"]),
    "technology": str(row["course_technology"]),
    "topic": str(row["course_topic"])
}, axis=1)

print(f"Total sections loaded: {len(files)}")
print(f"Total unique courses: {files['course_id'].nunique()}")
files[["unique_id", "course_name", "section_name"]].head()

Total sections loaded: 680
Total unique courses: 105


,unique_id,course_name,section_name
0,2-9,Introduction to Tableau,Introduction to Tableau
1,2-10,Introduction to Tableau,Tableau Functionalities
2,2-11,Introduction to Tableau,The Tableau Exercise
3,3-12,The Complete Data Visualization Course with Py...,Introduction
4,3-13,The Complete Data Visualization Course with Py...,Setting Up the Environments


## 4. Multi-Attribute Weighted Embedding Generation

Rather than treating all text equally, differential weights are assigned to balance search precision:
- **Course Name (Weight = 5):** Strongest semantic signal for broad course context
- **Section Name (Weight = 3):** Key signal for specific topic / chapter
- **Section Description (Weight = 2):** Granular detail
- **Course Technology (Weight = 1):** Language / framework alignment

In [192]:
def create_weighted_section_embedding(row):
    emb_course = model.encode(str(row["course_name"]), show_progress_bar=False) * 5
    emb_section = model.encode(str(row["section_name"]), show_progress_bar=False) * 3
    emb_desc = model.encode(str(row["section_description"]), show_progress_bar=False) * 2
    emb_tech = model.encode(str(row["course_technology"]), show_progress_bar=False) * 1
    
    combined = (emb_course + emb_section + emb_desc + emb_tech) / (5 + 3 + 2 + 1)
    return combined

print("Generating weighted embeddings for all sections...")
files["embedding"] = files.apply(create_weighted_section_embedding, axis=1)
print("Embeddings generated successfully. Sample shape:", files["embedding"].iloc[0].shape)

Generating weighted embeddings for all sections...
Embeddings generated successfully. Sample shape: (384,)


## 5. Upsert Vectors and Metadata into Pinecone

In [193]:
# Prepare batch upsert tuples: (id, vector, metadata)
vectors_to_upsert = [
    (row["unique_id"], row["embedding"].tolist(), row["metadata"])
    for _, row in files.iterrows()
]

# Batch upsert into Pinecone
batch_size = 100
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors=batch)

print(f"Successfully upserted {len(vectors_to_upsert)} section vectors into Pinecone.")
print("Updated index stats:", index.describe_index_stats())

Successfully upserted 680 section vectors into Pinecone.
Updated index stats: DescribeIndexStatsResponse(dimension=384, total_vector_count=680, metric='cosine', namespaces=1)


## 6. Granular Section-Level Semantic Search

Perform real-time semantic search queries against the vector index with cosine similarity threshold filtering and metadata retrieval.

In [194]:
def search_sections(query_text, top_k=5, score_threshold=0.3):
    query_vec = model.encode(query_text, show_progress_bar=False).tolist()
    results = index.query(vector=[query_vec], top_k=top_k, include_metadata=True)
    
    matches = [m for m in results["matches"] if m["score"] >= score_threshold]
    print(f"Query: '{query_text}' | Matches Found: {len(matches)}\n" + "-" * 70)
    for m in matches:
        meta = m.get("metadata", {})
        print(f"Score: {m['score']:.4f} | ID: {m['id']}")
        print(f"Course: {meta.get('course_name')}")
        print(f"Section: {meta.get('section_name')}")
        print(f"Topic: {meta.get('topic')} | Tech: {meta.get('technology')}")
        print(f"Description: {str(meta.get('section_description', ''))[:140]}...\n")
    return matches

# Test Query 1: Regularization in machine learning
search_sections("lasso regression and regularization in Python", top_k=3, score_threshold=0.4)

# Test Query 2: Unsupervised clustering
search_sections("clustering and customer segmentation", top_k=3, score_threshold=0.4)

Query: 'lasso regression and regularization in Python' | Matches Found: 3
----------------------------------------------------------------------
Score: 0.7804 | ID: 56-490
Course: Machine Learning with Ridge and Lasso Regression
Section: Regularization Basics
Topic: machine learning | Tech: python
Description: As an introduction to the course, we explore the concept of regularization and explain how it can be leveraged to prevent overfitting and mu...

Score: 0.7631 | ID: 56-492
Course: Machine Learning with Ridge and Lasso Regression
Section: Ridge and Lasso Regression – Practical Case
Topic: machine learning | Tech: python
Description: In this section, we will walk you through the implementation of ridge and lasso regression using sk-learn in Python. We apply these methods ...

Score: 0.6815 | ID: 56-491
Course: Machine Learning with Ridge and Lasso Regression
Section: Setting Up The Environment
Topic: machine learning | Tech: python
Description: If you’re new to programming with Pyt

[ScoredVector(id='33-335', score=0.707406461, values=[], metadata={'course_id': 33, 'course_name': 'Customer Analytics in Python', 'section_description': 'We continue with segmentation, this time focusing on a flat clustering technique, namely K-means. We’ll divide our customers into clusters using K-means and determine the customer segments. This is a crucial part of the customer analysis, as it is where we discover the most important characteristics, which define our customer groups.\r\r\r\r\r\r\r\r\r\r\r\n', 'section_name': 'K-means Clustering', 'technology': 'python', 'topic': 'financial analysis'}),
 ScoredVector(id='33-336', score=0.636961222, values=[], metadata={'course_id': 33, 'course_name': 'Customer Analytics in Python', 'section_description': 'In our final section of customer analytics, we’ll introduce a dimensionality reduction algorithm: Principal Components Analysis (PCA). We’ll combine K-means and PCA to obtain even better clustering results and gain insight about our 

## 7. Hierarchical Course-Level Aggregation & Search

Aggregate section-level vectors into course-level composite representations and store them in a dedicated Pinecone namespace (`courses`). This supports dual-level search: granular section retrieval and broad course discovery in a single vector index.

In [195]:
# Aggregate course data across sections
course_agg = files.groupby("course_id").agg({
    "course_name": "first",
    "course_slug": "first",
    "course_description": "first",
    "course_description_short": "first",
    "course_technology": "first",
    "course_topic": "first",
    "course_instructor_quote": "first"
}).reset_index()

# Robust encoder that safely handles null or missing values
def safe_encode(text, weight=1):
    if text is None or pd.isna(text) or not str(text).strip():
        return np.zeros(model.get_sentence_embedding_dimension())
    return model.encode(str(text), show_progress_bar=False) * weight

def create_course_composite_embedding(row):
    c_id = row["course_id"]
    emb_name = safe_encode(row["course_name"], weight=5)
    emb_desc = safe_encode(row["course_description"], weight=2)
    emb_tech = safe_encode(row["course_technology"], weight=2)
    emb_quote = safe_encode(row["course_instructor_quote"], weight=1)
    
    # Reuse section embeddings computed earlier for optimal efficiency
    section_embs = files[files["course_id"] == c_id]["embedding"].values
    if len(section_embs) > 0:
        mean_section_emb = np.mean(section_embs.tolist(), axis=0) * 4
    else:
        mean_section_emb = np.zeros(model.get_sentence_embedding_dimension())
    
    composite = np.mean([emb_name, emb_desc, emb_tech, emb_quote, mean_section_emb], axis=0)
    return composite

print("Generating composite course embeddings...")
course_agg["embedding"] = course_agg.apply(create_course_composite_embedding, axis=1)
print(f"Composite embeddings ready for {len(course_agg)} unique courses.")

Generating composite course embeddings...


C:\Users\Hp\AppData\Local\Temp\ipykernel_256\439694494.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return np.zeros(model.get_sentence_embedding_dimension())


Composite embeddings ready for 105 unique courses.


## 8. Upsert Course Vectors into Dedicated Namespace

In [196]:
# Prepare course-level vectors with metadata payload
course_vectors = [
    (
        str(row["course_id"]),
        row["embedding"].tolist(),
        {
            "course_name": str(row["course_name"]),
            "course_slug": str(row["course_slug"]),
            "topic": str(row["course_topic"]),
            "technology": str(row["course_technology"])
        }
    )
    for _, row in course_agg.iterrows()
]

# Upsert to the 'courses' namespace
index.upsert(vectors=course_vectors, namespace="courses")
print(f"Successfully upserted {len(course_vectors)} courses into namespace 'courses'.")

Successfully upserted 105 courses into namespace 'courses'.


## 9. Course-Level Discovery Search

In [197]:
def search_courses(query_text, top_k=5, score_threshold=0.25):
    query_vec = model.encode(query_text, show_progress_bar=False).tolist()
    results = index.query(vector=[query_vec], namespace="courses", top_k=top_k, include_metadata=True)
    
    matches = [m for m in results["matches"] if m["score"] >= score_threshold]
    print(f"Course Query: '{query_text}' | Matches Found: {len(matches)}\n" + "-" * 70)
    for m in matches:
        meta = m.get("metadata", {})
        print(f"Score: {m['score']:.4f} | Course ID: {m['id']}")
        print(f"Course Name: {meta.get('course_name')}")
        print(f"Topic: {meta.get('topic')} | Technology: {meta.get('technology')}\n")
    return matches

# Test Course Search
search_courses("deep learning neural networks", top_k=3, score_threshold=0.25)
search_courses("data visualization with tableau or power bi", top_k=3, score_threshold=0.25)

Course Query: 'deep learning neural networks' | Matches Found: 3
----------------------------------------------------------------------
Score: 0.7197 | Course ID: 31
Course Name: Deep Learning with TensorFlow
Topic: machine learning | Technology: python

Score: 0.6532 | Course ID: 32
Course Name: Deep Learning with TensorFlow 2
Topic: machine learning | Technology: python

Score: 0.5574 | Course ID: 41
Course Name: Convolutional Neural Networks with TensorFlow in Python
Topic: machine learning | Technology: python

Course Query: 'data visualization with tableau or power bi' | Matches Found: 3
----------------------------------------------------------------------
Score: 0.7259 | Course ID: 2
Course Name: Introduction to Tableau
Topic: data visualization | Technology: tableau

Score: 0.6651 | Course ID: 3
Course Name: The Complete Data Visualization Course with Python, R, Tableau, and Excel
Topic: data visualization | Technology: python

Score: 0.6551 | Course ID: 70
Course Name: Buildin

[ScoredVector(id='2', score=0.725869775, values=[], metadata={'course_name': 'Introduction to Tableau', 'course_slug': 'tableau', 'technology': 'tableau', 'topic': 'data visualization'}),
 ScoredVector(id='3', score=0.665059328, values=[], metadata={'course_name': 'The Complete Data Visualization Course with Python, R, Tableau, and Excel', 'course_slug': 'data-visualization', 'technology': 'python', 'topic': 'data visualization'}),
 ScoredVector(id='70', score=0.655102253, values=[], metadata={'course_name': 'Building Business Reports Using Power BI', 'course_slug': 'building-business-reports-using-power-bi', 'technology': 'powerbi', 'topic': 'data analysis'})]